In [1]:
import numpy as np
import pandas as pd
import snowflake.connector
from datetime import datetime, timedelta
import requests
import io
from scipy.optimize import linprog
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus, value

In [2]:
df_plan_compra_general = pd.read_csv(r"C:\Users\juane\Downloads\PLAN_TEST.csv")

In [3]:
df_plan_compra_general["WAREHOUSE_PRODUCT_ID"]=df_plan_compra_general["Warehouseid"].astype(str)+"-"+df_plan_compra_general["Sku Id"].astype(str)
df_plan_compra_general["ADJUSTED_QUANTITY"] = df_plan_compra_general["Purchase Quantity"]
df_plan_compra_general["Variable"] = df_plan_compra_general.apply(lambda row: LpVariable(f"Q_{row['WAREHOUSE_PRODUCT_ID']}", lowBound=row["ADJUSTED_QUANTITY"]), axis=1)
df_plan_compra_general = df_plan_compra_general.reset_index(drop=True)

In [4]:
df_plan_compra_general["Average Forecasts for Next 7 Days"] = df_plan_compra_general["Average Forecasts for Next 7 Days"].replace(0, 0.001)

In [5]:
supplier_restrictions = {
    316: [ #STLTH
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 30, 'max_quantity': 5000}
    ],
    57: [ #Vive Agro
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 25, 'max_quantity': 5000}
    ],
    178: [ #Kampos carnicos
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 10, 'max_quantity': 5000},
    ],
    313: [ #Notco
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 20, 'max_quantity': 5000}
    ],
    166: [ #Selvati
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 20, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 20, 'max_quantity': 5000}
    ],
   210: [ #Pod Salt
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 56, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 80, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 50, 'max_quantity': 5000}, 
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 50, 'max_quantity': 5000}
   ],
    168:[ #Glucloud
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 50, 'max_quantity': 5000}
    ],
    4:[ #Inversiones Liev
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 10, 'max_quantity': 5000}
    ],
    200:[ #Arte Dolce
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 40, 'max_quantity': 5000}
    ],
    135: [ #Sr Wok
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 80, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 18, 'max_quantity': 5000}
    ],
    185:[ #RELX
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 80, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 25, 'max_quantity': 5000}
    ],
    318: [ #Evapify
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 80, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 30, 'max_quantity': 5000}
    ],
    77:[ #Pan pa ya
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_value': 70000, 'max_value': 10000000}
    ],
    119 :[ #Pixie
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 35, 'min_value': 800000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 3, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 5, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 7, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 36, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 37, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 40, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 42, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 43, 'min_value': 350000, 'max_value': 10000000}
    ],
    
}

In [6]:
def optimize(df, restriction, restriction_type, objective):
    if restriction_type == 'quantity' and objective == "increment":
        model = LpProblem("Optimization_Plan", LpMinimize)
        model += lpSum([var * (1 / avg_forecast) for var, avg_forecast in zip(df["Variable"], df["Average Forecasts for Next 7 Days"])]), "Minimize_Increments"
        model += lpSum(df["Variable"]) >= restriction, "Sum_Adjusted_Quantity"
        model.solve()
        df["ADJUSTED_QUANTITY"] = [var.varValue for var in df["Variable"]]
    elif restriction_type == 'quantity' and objective == "decrease":
        model = LpProblem("Optimization_Plan", LpMaximize)
        model += lpSum([var * (1 / avg_forecast) for var, avg_forecast in zip(df["Variable"], df["Average Forecasts for Next 7 Days"])]), "Maximize_Increments"
        model += lpSum(df["Variable"]) <= restriction, "Sum_Adjusted_Quantity"
        model.solve()
        df["ADJUSTED_QUANTITY"] = [var.varValue for var in df["Variable"]]
    elif restriction_type == 'monetary' and objective == "increment":
        model = LpProblem("Optimization_Plan", LpMinimize)
        model += lpSum([var * (1 / avg_forecast) for var, avg_forecast in zip(df["Variable"], df["Average Forecasts for Next 7 Days"])]), "Minimize_Increments"
        lpSum([var * price for var, price in zip(df["Variable"], df['Supplier Price Avg'])]) >= restriction, "Sum_Adjusted_Quantity"
        model.solve()
        df["ADJUSTED_QUANTITY"] = [var.varValue for var in df["Variable"]]
    elif restriction_type == 'monetary' and objective == "decrease":
        model = LpProblem("Optimization_Plan", LpMaximize)
        model += lpSum([var * (1 / avg_forecast) for var, avg_forecast in zip(df["Variable"], df["Average Forecasts for Next 7 Days"])]), "Maximize_Increments"
        lpSum([var * price for var, price in zip(df["Variable"], df['Supplier Price Avg'])]) <= restriction, "Sum_Adjusted_Quantity"
        model.solve()
        df["ADJUSTED_QUANTITY"] = [var.varValue for var in df["Variable"]]
    return df 

In [7]:
df_plan_compra_general['VIOLATION'] = False
df_plan_compra_general['VIOLATION_DESCRIPTION'] = ''
for supplier, restrictions in supplier_restrictions.items():
    supplier_df = df_plan_compra_general[df_plan_compra_general['Primary Supplier Id'] == supplier]
    if supplier_df.empty:
        continue
    for restriction in restrictions:
        restriction_type = restriction['type']
        level = restriction['level']

        if level == 'warehouse':
            group_col = 'Warehouseid'
        elif level == 'city':
            group_col = 'City ID'
        elif level == 'product':
            group_col = 'Sku Id'
        else: 
            group_col = None

        if group_col:
            if group_col not in supplier_df.columns:
                continue
            grouped = supplier_df.groupby(group_col)

            if not grouped.groups:
                continue

            for key, group in grouped:
                if restriction_type == 'quantity':
                    total_quantity = group['ADJUSTED_QUANTITY'].sum()
                    if total_quantity < restriction['min_quantity'] or total_quantity > restriction['max_quantity']:
                        df_plan_compra_general.loc[group.index, 'VIOLATION'] = True
                        df_plan_compra_general.loc[group.index, 'VIOLATION_DESCRIPTION'] = f'{restriction_type} violation at {level} level for {group_col}={key}'
                        if total_quantity < restriction['min_quantity']:
                            min_quantity = restriction.get('min_quantity', 0)
                            optimize(group, min_quantity, restriction_type, "increment")
                        elif total_quantity > restriction['max_quantity']:
                            max_quantity = restriction.get('max_quantity', 0)
                            optimize(group, max_quantity, restriction_type, "decrease")

                elif restriction_type == 'monetary':
                    total_value = (group['ADJUSTED_QUANTITY'] * group['Supplier Price Avg']).sum()
                    if total_value < restriction['min_value'] or total_value > restriction['max_value']:
                        df_plan_compra_general.loc[group.index, 'VIOLATION'] = True
                        df_plan_compra_general.loc[group.index, 'VIOLATION_DESCRIPTION'] = f'{restriction_type} violation at {level} level for {group_col}={key}'
                        if total_value < restriction['min_value']:
                            min_value = restriction.get('min_value', 0)
                            optimize(group, min_value, restriction_type, "increment")
                        elif total_value > restriction['max_value']:
                            max_value = restriction.get('max_value', 0)
                            optimize(group, max_value, restriction_type, "decrease")
        else: 
            if restriction_type == 'quantity':
                total_quantity = supplier_df['ADJUSTED_QUANTITY'].sum()
                if total_quantity < restriction['min_quantity'] or total_quantity > restriction['max_quantity']:
                    df_plan_compra_general.loc[supplier_df.index, 'violation'] = True
                    df_plan_compra_general.loc[supplier_df.index, 'violation_description'] = f'{restriction_type} violation at total level'
                    if total_quantity < restriction['min_quantity']:
                        min_quantity = restriction.get('min_quantity', 0)
                        optimize(group, min_quantity, restriction_type, "increment")
                    elif total_quantity > restriction['max_quantity']:
                            max_quantity = restriction.get('max_quantity', 0)
                            optimize(group, max_quantity, restriction_type, "decrease")           
            elif restriction_type == 'monetary':
                total_value = (supplier_df['ADJUSTED_QUANTITY'] * supplier_df['Supplier Price Avg']).sum()
                if total_value < restriction['min_value'] or total_value > restriction['max_value']:
                    df_plan_compra_general.loc[supplier_df.index, 'violation'] = True
                    df_plan_compra_general.loc[supplier_df.index, 'violation_description'] = f'{restriction_type} violation at total level'
                    if total_value < restriction['min_value']:
                            min_value = restriction.get('min_value', 0)
                            optimize(group, min_value, restriction_type, "increment")
                    elif total_value > restriction['max_value']:
                        max_value = restriction.get('max_value', 0)
                        optimize(group, max_value, restriction_type, "decrease")
                        
df_plan_compra_general.to_csv("C:/Users/juane/Downloads/PURCHASE_PLAN_COLOMBIA.csv", index=False)

restriccion: 25
Valores optimizados de las variables de decisión:
Q_8_7457: 3.0
Q_8_4817: 3.0
Q_8_12914: 2.0
Q_8_11834: 2.0
Q_8_9186: 1.0
Q_8_8875: 1.0
Q_8_11574: 0.0
Q_8_11868: 13.0
Q_8_12356: 0.0
Q_8_6550: 0.0
Q_8_8883: 0.0
     Warehouseid  Sku Id  Primary Supplier Id  Purchase Quantity  \
621            8    7457                   57                  3   
624            8    4817                   57                  3   
630            8   12914                   57                  2   
632            8   11834                   57                  2   
679            8    9186                   57                  1   
681            8    8875                   57                  1   
708            8   11574                   57                  0   
709            8   11868                   57                  0   
711            8   12356                   57                  0   
717            8    6550                   57                  0   
719            8    8883  